# Funding signal model experimentation, continued

Audit of the frozen funding predictor before a strategy is built on it. The question is what the 0.7 rank IC is once decomposed, whether it survives honest past-only training, and whether it survives trading costs. Also builds the causal out-of-fold prediction table the strategy work runs on. Every prediction is out of fold.

## Feature frame

Funding and premium events, the causal feature chain, kline aggregates, interaction and basket features. Target is realized cumulative funding over the next 24 8h prints. 32058 rows, 10 symbols, 2022 to 2024.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.pipeline import Pipeline

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "research").is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import research.features as F
import research.cv as CV
from research.models import _with_symbol_dummies

DATA = REPO / "data" / "binance_historical"
SYMBOLS = ["BTCUSDT","ETHUSDT","SOLUSDT","BNBUSDT","XRPUSDT",
           "DOGEUSDT","ADAUSDT","LINKUSDT","AVAXUSDT","LTCUSDT"]
HORIZON = 24
FUNDING_COLS = ["symbol","tag","ts_ms","interval_h","funding_rate"]
PREMIUM_COLS = ["ts_ms","open","high","low","close","volume","close_ms",
                "quote_vol","trades","taker_buy_base","taker_buy_quote","ignore"]

In [ ]:
def load_funding(sym):
    d = DATA/sym/"futures"/"funding"
    files = sorted(d.glob(f"{sym}-fundingRate-*.csv"))
    df = pd.concat([pd.read_csv(f,header=None,names=FUNDING_COLS) for f in files],ignore_index=True)
    df["ts"]=pd.to_datetime(df["ts_ms"],unit="ms",utc=True); df["symbol"]=sym
    df=df[["symbol","ts","funding_rate"]].rename(columns={"funding_rate":"realized_funding"})
    return df.sort_values("ts").drop_duplicates("ts").reset_index(drop=True)

def load_premium(sym):
    d = DATA/sym/"futures"/"premiumindex"; files=sorted(d.glob("*.csv"))
    if not files: return pd.DataFrame(columns=["ts_ms","close"])
    parts=[pd.read_csv(f,header=None,names=PREMIUM_COLS)[["ts_ms","close"]] for f in files]
    df=pd.concat(parts,ignore_index=True).sort_values("ts_ms").drop_duplicates("ts_ms").reset_index(drop=True)
    df["ts"]=pd.to_datetime(df["ts_ms"],unit="ms",utc=True); return df

def average_premium_up_to(premium, funding_ts, interval_hours=8):
    if premium.empty: return pd.Series(np.nan,index=range(len(funding_ts)))
    p=premium.sort_values("ts").reset_index(drop=True); win=pd.Timedelta(hours=interval_hours)
    out=np.full(len(funding_ts),np.nan)
    ts_vals=pd.to_datetime(funding_ts,utc=True).to_numpy()
    p_ts=p["ts"].to_numpy(); p_close=p["close"].to_numpy(dtype=float)
    for i,end in enumerate(ts_vals):
        beg=end-win; lo=np.searchsorted(p_ts,beg,side="right"); hi=np.searchsorted(p_ts,end,side="right")
        if hi>lo: out[i]=p_close[lo:hi].mean()
    return pd.Series(out)

parts=[]
for sym in SYMBOLS:
    f=load_funding(sym); p=load_premium(sym)
    f["premium"]=average_premium_up_to(p,f["ts"]).to_numpy(); parts.append(f)
events=pd.concat(parts,ignore_index=True)
print("events:", events.shape)

events: (32955, 4)


In [3]:
feat=events.copy()
for k in (1,2,3): feat=F.add_funding_lag(feat,k)
feat=F.add_funding_ewma(feat,halflife=2); feat=F.add_funding_ewma(feat,halflife=6)
feat=F.add_funding_vol(feat,window=8); feat=F.add_clamp_distance(feat)
feat=F.add_premium_trend(feat,span=3); feat=F.add_cross_symbol_spread(feat,reference="BTCUSDT")
feat=F.add_basket_spread(feat); feat=F.add_time_features(feat)
feat=F.add_funding_mean_window(feat,window=90); feat=F.add_funding_sign_window(feat,window=90)
feat=F.add_funding_vol_rank(feat,vol_window=30,rank_window=180)
kline_agg=pd.read_pickle(REPO/"research"/"results"/"kline_aggregates_8h.pkl")
feat=feat.merge(kline_agg,on=["symbol","ts"],how="left")
feat["taker_imb_x_ewma2"]=feat["taker_imbalance"]*feat["funding_ewma_h2"]
feat["taker_imb_x_lag1"]=feat["taker_imbalance"]*feat["funding_lag1"]
feat["vol1m_x_clamp"]=feat["realized_vol_1m"]*feat["clamp_distance"]
feat["range_x_ewma2"]=feat["high_low_range"]*feat["funding_ewma_h2"]
feat["regime_x_lag1"]=feat["funding_sign_w90"]*feat["funding_lag1"]
feat["basket_x_lag1"]=feat["basket_spread"]*feat["funding_lag1"]
feat["lag1_sq"]=feat["funding_lag1"]**2
feat=F.add_basket_zscore(feat,source="funding_lag1")
feat=F.add_basket_zscore(feat,source="funding_ewma_h6")
feat=F.add_basket_rank(feat,source="funding_lag1")
feat["basket_z_x_lag1"]=feat["basket_z_funding_lag1"]*feat["funding_lag1"]
feat["basket_rank_x_lag1"]=feat["basket_rank_funding_lag1"]*feat["funding_lag1"]
feat=F.add_cum_target(feat,horizon=HORIZON)
FCOLS=[c for c in feat.columns if c not in ("symbol","ts","realized_funding","premium","realized_cum")]
feat=feat.dropna(subset=FCOLS+["realized_funding","realized_cum"]).reset_index(drop=True)
feat["year"]=feat["ts"].dt.year
print("feat_v5_clean:", feat.shape, " features:", len(FCOLS))

feat_v5_clean: (32058, 37)  features: 31


## Out-of-fold predictions

Two prediction sets from the same purged folds. One trains on both sides of each test block. The other trains only on the past, the honest deployment condition. Persistence is 24 times the last print. Elastic is the pooled net with symbol dummies. GBM is early stopped.

In [4]:
def rank_ic(y,p):
    if len(y)<10 or np.std(p)==0: return np.nan
    return spearmanr(p,y).correlation

def elastic_pred(train,test,cols,target="realized_cum"):
    Xtr,_=_with_symbol_dummies(train,cols)
    pipe=Pipeline([("sc",StandardScaler()),("m",ElasticNet(alpha=1e-4,l1_ratio=0.7,max_iter=20000))])
    pipe.fit(Xtr,train[target].to_numpy()); Xte,_=_with_symbol_dummies(test,cols); return pipe.predict(Xte)

def gbm_pred(train,test,cols,target="realized_cum"):
    import lightgbm as lgb
    Xtr,c=_with_symbol_dummies(train,cols); y=train[target].to_numpy(); n=len(y); sp=int(n*0.8)
    p=dict(objective="regression",verbose=-1,feature_fraction=0.9,bagging_fraction=0.9,
           bagging_freq=5,learning_rate=0.02,num_leaves=15,min_data_in_leaf=500)
    dtr=lgb.Dataset(Xtr[:sp],label=y[:sp],feature_name=c); dval=lgb.Dataset(Xtr[sp:],label=y[sp:],reference=dtr)
    b=lgb.train(p,dtr,num_boost_round=3000,valid_sets=[dval],
                callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(0)])
    Xte,_=_with_symbol_dummies(test,cols); return b.predict(Xte,num_iteration=b.best_iteration)

def gen_oof(causal=False, do_gbm=True, target="realized_cum"):
    rows=[]
    for fi,(tr,te) in enumerate(CV.walk_forward_splits(feat,n_folds=5,horizon=HORIZON,embargo=5)):
        train=feat.iloc[tr]; test=feat.iloc[te]
        if causal:
            train=train[train["ts"]<test["ts"].min()]
            if len(train)<2000: continue
        out=test[["symbol","ts","realized_funding","realized_cum","year","funding_lag1"]].copy()
        out["pred_elastic"]=elastic_pred(train,test,FCOLS,target)
        out["pred_persist"]=(HORIZON*test["funding_lag1"]).to_numpy()
        out["fold"]=fi
        if do_gbm: out["pred_gbm"]=gbm_pred(train,test,FCOLS,target)
        rows.append(out)
    return pd.concat(rows,ignore_index=True)

oof  = gen_oof(causal=False, do_gbm=True)
oofc = gen_oof(causal=True,  do_gbm=True)
print("two-sided oof:", oof.shape, " causal oof:", oofc.shape)

two-sided oof: (22448, 10)  causal oof: (22448, 10)


## What the 0.7 is made of

Pooled rank IC mixes two abilities. Cross-sectional IC ranks the 10 symbols against each other at each timestamp, picking symbols. Time-series IC ranks within a symbol over time, timing entries. Lift is over persistence.

In [5]:
def pooled_ic(o,c): return rank_ic(o["realized_cum"].to_numpy(),o[c].to_numpy())
def xsec_ic(o,c): return o.groupby("ts").apply(lambda g: rank_ic(g["realized_cum"].to_numpy(),g[c].to_numpy())).mean()
def tsec_ic(o,c): return o.groupby("symbol").apply(lambda g: rank_ic(g["realized_cum"].to_numpy(),g[c].to_numpy())).mean()

rows=[]
for name,fn in [("pooled rank IC",pooled_ic),("cross-sectional IC (avg/ts)",xsec_ic),("per-symbol time-series IC",tsec_ic)]:
    e,g,p=fn(oof,"pred_elastic"),fn(oof,"pred_gbm"),fn(oof,"pred_persist")
    rows.append({"metric":name,"elastic":e,"gbm":g,"persist":p,"e_lift":e-p,"g_lift":g-p})
pd.DataFrame(rows).set_index("metric").round(4)

,elastic,gbm,persist,e_lift,g_lift
metric,,,,,
pooled rank IC,0.7854,0.8204,0.7343,0.0511,0.0862
cross-sectional IC (avg/ts),0.5204,0.5297,0.4149,0.1055,0.1149
per-symbol time-series IC,0.7862,0.7877,0.7100,0.0762,0.0777


Cross-sectional lift beats time-series lift. The model picks symbols better than it times one.

In [6]:
# per-symbol time-series rank IC, elastic vs persistence
r=[]
for s in sorted(feat["symbol"].unique()):
    ge=oof[oof.symbol==s]
    e=rank_ic(ge["realized_cum"].to_numpy(),ge["pred_elastic"].to_numpy())
    p=rank_ic(ge["realized_cum"].to_numpy(),ge["pred_persist"].to_numpy())
    r.append({"symbol":s,"elastic":e,"persist":p,"lift":e-p})
pd.DataFrame(r).set_index("symbol").round(3)

,elastic,persist,lift
symbol,,,
ADAUSDT,0.770,0.690,0.080
AVAXUSDT,0.823,0.743,0.079
BNBUSDT,0.683,0.613,0.071
BTCUSDT,0.841,0.765,0.076
DOGEUSDT,0.755,0.674,0.081
ETHUSDT,0.830,0.757,0.073
LINKUSDT,0.772,0.694,0.079
LTCUSDT,0.827,0.743,0.084
SOLUSDT,0.765,0.727,0.038


In [7]:
# causal past-only training vs two-sided: how much of the IC is future leakage
print(f"pooled IC     causal={pooled_ic(oofc,'pred_elastic'):.4f}   two-sided={pooled_ic(oof,'pred_elastic'):.4f}")
print(f"per-sym TS    causal={tsec_ic(oofc,'pred_elastic'):.4f}   two-sided={tsec_ic(oof,'pred_elastic'):.4f}   persist TS={tsec_ic(oof,'pred_persist'):.4f}")

pooled IC     causal=0.6917   two-sided=0.7854
per-sym TS    causal=0.6990   two-sided=0.7862   persist TS=0.7100


Training only on the past drops pooled IC by about 0.09 and pulls the per-symbol timing IC to at or below persistence. The timing edge is mostly an artifact of training on future data.

In [8]:
# IC by year (two-sided)
r=[]
for y in sorted(oof.year.unique()):
    g=oof[oof.year==y]
    e,gg,p=pooled_ic(g,"pred_elastic"),pooled_ic(g,"pred_gbm"),pooled_ic(g,"pred_persist")
    r.append({"year":y,"elastic":e,"gbm":gg,"persist":p,"e_lift":e-p,"n":len(g)})
pd.DataFrame(r).set_index("year").round(4)

,elastic,gbm,persist,e_lift,n
year,,,,,
2022,0.7195,0.7415,0.4322,0.2873,750
2023,0.7154,0.7733,0.6595,0.0559,10950
2024,0.8298,0.8439,0.7899,0.0399,10748


In [9]:
# IC by horizon: refit elastic per horizon, lift over persistence
def target_h(frame,h):
    f=frame.sort_values(["symbol","ts"]).copy()
    f["y_h"]=f.groupby("symbol")["realized_funding"].transform(lambda s: s.rolling(h,min_periods=h).sum().shift(-(h-1)))
    return f
r=[]
for h in (1,3,6,12,24,48):
    fh=target_h(feat,h).dropna(subset=["y_h"]).reset_index(drop=True)
    prs=[]
    for fi,(tr,te) in enumerate(CV.walk_forward_splits(fh,n_folds=5,horizon=h,embargo=5)):
        tr_d,te_d=fh.iloc[tr],fh.iloc[te]
        Xtr,_=_with_symbol_dummies(tr_d,FCOLS)
        pipe=Pipeline([("sc",StandardScaler()),("m",ElasticNet(alpha=1e-4,l1_ratio=0.7,max_iter=20000))])
        pipe.fit(Xtr,tr_d["y_h"].to_numpy()); Xte,_=_with_symbol_dummies(te_d,FCOLS)
        o=te_d[["y_h","funding_lag1"]].copy(); o["pe"]=pipe.predict(Xte); o["pp"]=h*te_d["funding_lag1"].to_numpy(); prs.append(o)
    O=pd.concat(prs,ignore_index=True)
    e=rank_ic(O["y_h"],O["pe"]); p=rank_ic(O["y_h"],O["pp"])
    r.append({"horizon":h,"elastic":e,"persist":p,"lift":e-p,"n":len(O)})
pd.DataFrame(r).set_index("horizon").round(4)

,elastic,persist,lift,n
horizon,,,,
1,0.8384,0.7903,0.0481,22448
3,0.8774,0.8087,0.0687,22428
6,0.8324,0.7943,0.0381,22408
12,0.7935,0.7689,0.0246,22368
24,0.7883,0.7359,0.0524,22288
48,0.7305,0.6880,0.0425,22118


Lift does not grow with horizon. Holding longer does not make the model worth more over persistence.

In [10]:
# IC on the magnitude subset a strategy actually trades
r=[]
for hurdle in (0.0017,0.0034,0.0068):
    g=oof[oof["pred_elastic"].abs()>hurdle]
    if len(g)<50: continue
    r.append({"hurdle_bps":hurdle*1e4,"n":len(g),"share":len(g)/len(oof),
              "elastic_IC":rank_ic(g["realized_cum"],g["pred_elastic"]),
              "persist_IC":rank_ic(g["realized_cum"],g["pred_persist"]),
              "dir_acc":np.mean(np.sign(g["pred_elastic"])==np.sign(g["realized_cum"]))})
pd.DataFrame(r).set_index("hurdle_bps").round(3)

,n,share,elastic_IC,persist_IC,dir_acc
hurdle_bps,,,,,
17.0,10715,0.477,0.783,0.736,0.970
34.0,3698,0.165,0.654,0.657,0.983
68.0,545,0.024,0.652,0.693,0.991


The finding that matters. Above the round-trip cost hurdle, where a strategy actually trades, the elastic ties persistence. Its lift lives in the near-zero rows nothing trades.

## Leakage and methodology

In [11]:
# Purge width, measured. The splitter is timestamp-positional, so horizon=24 is 24 real timestamps.
r=[]
for f,(tr,te) in enumerate(CV.walk_forward_splits(feat,n_folds=5,horizon=24,embargo=5)):
    ttr=feat.iloc[tr]["ts"]; tte=feat.iloc[te]["ts"]
    below=ttr[ttr<tte.min()].max(); above=ttr[ttr>tte.max()].min()
    gl=(tte.min()-below).total_seconds()/3600/8 if pd.notna(below) else None
    gh=(above-tte.max()).total_seconds()/3600/8 if pd.notna(above) else None
    r.append({"fold":f,"test_start":tte.min().date(),"test_end":tte.max().date(),
              "n_train":len(tr),"n_test":len(te),"purge_below_intervals":gl,"embargo_above_intervals":gh})
pd.DataFrame(r).set_index("fold")

,test_start,test_end,n_train,n_test,purge_below_intervals,embargo_above_intervals
fold,,,,,,
0,2022-12-07,2023-05-05,27038,4490,25.0,30.0
1,2023-05-05,2023-10-02,27038,4490,25.0,30.0
2,2023-10-02,2024-02-28,27038,4490,25.0,30.0
3,2024-02-29,2024-07-27,27040,4488,25.0,30.0
4,2024-07-27,2024-12-24,27328,4490,25.0,NaN


The measured purge gap is 25 intervals, about 8 days, matching the 24-print label. The high IC is not a too-narrow purge. Training is two-sided though, each fold trains on rows before and after its test block. Sound for a signal study, look-ahead for a backtest, so the prediction table is causal.

In [12]:
# Export parity: the frozen JSON is a Ridge on robustness-selected features, fit on the full window.
fold_coefs=[]
for fi,(tr,_) in enumerate(CV.walk_forward_splits(feat,n_folds=5,horizon=HORIZON,embargo=5)):
    tr_d=feat.iloc[tr]; Xtr,_=_with_symbol_dummies(tr_d,FCOLS)
    pipe=Pipeline([("sc",StandardScaler()),("m",ElasticNet(alpha=1e-4,l1_ratio=0.7,max_iter=20000))])
    pipe.fit(Xtr,tr_d["realized_cum"].to_numpy()); fold_coefs.append(pipe["m"].coef_)
fc=np.array(fold_coefs); nzf=(np.abs(fc)>1e-10).mean(axis=0); nfeat=len(FCOLS)
sel=[f for f,r in zip(FCOLS,nzf[:nfeat]>=0.8) if r]
rows=[]
for fi,(tr,te) in enumerate(CV.walk_forward_splits(feat,n_folds=5,horizon=HORIZON,embargo=5)):
    tr_d,te_d=feat.iloc[tr],feat.iloc[te]
    Xtr,_=_with_symbol_dummies(tr_d,sel); Xte,_=_with_symbol_dummies(te_d,sel)
    pipe=Pipeline([("sc",StandardScaler()),("m",Ridge(alpha=1.0))]); pipe.fit(Xtr,tr_d["realized_cum"].to_numpy())
    o=te_d[["realized_cum"]].copy(); o["pred"]=pipe.predict(Xte); rows.append(o)
R=pd.concat(rows,ignore_index=True)
print(f"exported-form (Ridge on {len(sel)} selected feats) OOF pooled IC = {rank_ic(R['realized_cum'],R['pred']):.4f}")
print("The frozen artifact itself is fit on the full 2022-2024 window, so feeding it to a backtest is look-ahead. Use the OOF table.")

exported-form (Ridge on 17 selected feats) OOF pooled IC = 0.7919
The frozen artifact itself is fit on the full 2022-2024 window, so feeding it to a backtest is look-ahead. Use the OOF table.


In [13]:
# Feature drift: PSI of fold-1 train vs each test fold, top-5 coefficient features
def psi(a,b,bins=10):
    q=np.quantile(a,np.linspace(0,1,bins+1)); q[0]=-np.inf; q[-1]=np.inf
    ea=np.histogram(a,q)[0]/len(a)+1e-6; eb=np.histogram(b,q)[0]/len(b)+1e-6
    return float(np.sum((eb-ea)*np.log(eb/ea)))
top=[FCOLS[i] for i in np.argsort(-np.abs(fc.mean(0)[:nfeat]))[:5]]
folds=list(CV.walk_forward_splits(feat,n_folds=5,horizon=HORIZON,embargo=5))
tr0=feat.iloc[folds[0][0]]
psis={f:[psi(tr0[f].to_numpy(),feat.iloc[te][f].to_numpy()) for _,te in folds] for f in top}
pd.DataFrame(psis,index=[f"f{i}_test" for i in range(len(folds))]).T.round(2)

,f0_test,f1_test,f2_test,f3_test,f4_test
clamp_distance,0.16,0.48,0.87,0.32,0.07
vol1m_x_clamp,0.21,0.75,0.94,0.25,0.06
premium_trend_s3,0.04,0.05,0.04,0.01,0.01
funding_lag1,0.13,0.38,0.51,0.26,0.07
funding_ewma_h2,0.17,1.46,0.88,0.43,0.09


PSI over 0.25 is a major shift. Every funding-derived feature blows past it. One frozen scaler is badly miscalibrated across regimes, so the scaler is refit per fold.

## Net-carry retrain

Target swapped to net carry, cumulative funding minus the both-leg round trip priced per symbol and week. If the model cannot rank net carry, no strategy shape rescues it.

In [14]:
ct=pd.read_csv(REPO/"data"/"cost_model"/"cost_table.csv")
ct["week"]=pd.to_datetime(ct["week_start"],utc=True)
SPOT_FEE=10.0
ct["rt_bps"]=2*(ct["taker_fee_bps"]+ct["half_spread_bps"]) + 2*(SPOT_FEE+ct["half_spread_bps"])
feat["week"]=feat["ts"].dt.to_period("W-SUN").dt.start_time.dt.tz_localize("UTC")
med=ct.groupby("symbol")["rt_bps"].median()
feat=feat.merge(ct[["symbol","week","rt_bps"]],on=["symbol","week"],how="left")
feat["rt_bps"]=feat["rt_bps"].fillna(feat["symbol"].map(med)).fillna(ct["rt_bps"].median())
feat["net_carry"]=feat["realized_cum"]-feat["rt_bps"]/1e4
print(f"round trip both legs: median {feat['rt_bps'].median():.1f} bps")
print(f"rows with positive net carry (long clears cost): {np.mean(feat['net_carry']>0):.3f}")

on=gen_oof(causal=False, do_gbm=False, target="net_carry")
on=on.merge(feat[["symbol","ts","rt_bps","net_carry"]],on=["symbol","ts"],how="left")
on["persist_net"]=on["pred_persist"]-on["rt_bps"]/1e4
def ic2(c1,c2): return rank_ic(on[c1].to_numpy(),on[c2].to_numpy())
print(f"pooled IC   elastic={ic2('net_carry','pred_elastic'):.4f}  persist={ic2('net_carry','persist_net'):.4f}")
print(f"sign dir_acc elastic={np.mean(np.sign(on['pred_elastic'])==np.sign(on['net_carry'])):.3f}  persist={np.mean(np.sign(on['persist_net'])==np.sign(on['net_carry'])):.3f}")

round trip both legs: median 29.4 bps
rows with positive net carry (long clears cost): 0.135
pooled IC   elastic=0.7748  persist=0.7346
sign dir_acc elastic=0.897  persist=0.891


The model still ranks net carry with a small lift and ties persistence on the sign. Only about 13% of rows have positive net carry, so a long carry book is idle most of the time.

## Decision gate

Proceed to strategy work if the causal per-symbol timing lift is positive in at least two of three years and the net-carry retrain keeps a positive lift.

In [15]:
def tsic_g(o,c): return o.groupby("symbol").apply(lambda g: rank_ic(g["realized_cum"].to_numpy(),g[c].to_numpy())).mean()
def xsic_g(o,c): return o.groupby("ts").apply(lambda g: rank_ic(g["realized_cum"].to_numpy(),g[c].to_numpy())).mean()
r=[]
for y in sorted(oofc.year.unique()):
    g=oofc[oofc.year==y]
    e,p=tsic_g(g,"pred_elastic"),tsic_g(g,"pred_persist")
    xe,xp=xsic_g(g,"pred_elastic"),xsic_g(g,"pred_persist")
    r.append({"year":y,"elastic_TS":e,"persist_TS":p,"TS_lift":e-p,"xsec_lift":xe-xp})
e,p=tsic_g(oofc,"pred_elastic"),tsic_g(oofc,"pred_persist")
r.append({"year":"ALL","elastic_TS":e,"persist_TS":p,"TS_lift":e-p,"xsec_lift":np.nan})
pd.DataFrame(r).set_index("year").round(3)

,elastic_TS,persist_TS,TS_lift,xsec_lift
year,,,,
2022,0.236,0.067,0.169,0.297
2023,0.541,0.621,-0.081,0.069
2024,0.792,0.768,0.024,0.064
ALL,0.699,0.710,-0.011,NaN


Causal timing lift is positive in two of three years, but 2022 is thin and 2023 is negative, and pooled is negative. Timing is marginal. Cross-sectional lift is positive every year. Build the cross-sectional strategy first and drive the backtest from causal predictions.

## Profitability

Fees dominate, so the strategy is low turnover. The arithmetic from the real cost table.

In [16]:
g=ct.groupby("symbol").agg(hs=("half_spread_bps","median"),fee=("taker_fee_bps","median")).reset_index()
g["perp_rt"]=2*(g["fee"]+g["hs"]); g["spot_rt"]=2*(SPOT_FEE+g["hs"]); g["both_rt"]=g["perp_rt"]+g["spot_rt"]
print("per-symbol round trip, both legs, taker VIP0, spot half-spread copied from futures:")
display(g.round(2).set_index("symbol"))
print(f"median both-legs round trip: {g['both_rt'].median():.1f} bps  (plan placeholder assumed ~34)")

per-symbol round trip, both legs, taker VIP0, spot half-spread copied from futures:


,hs,fee,perp_rt,spot_rt,both_rt
symbol,,,,,
ADAUSDT,1.37,4.0,10.74,22.74,33.47
AVAXUSDT,0.36,4.0,8.72,20.72,29.44
BNBUSDT,0.21,4.0,8.41,20.41,28.82
BTCUSDT,0.02,4.0,8.03,20.03,28.06
DOGEUSDT,0.67,4.0,9.33,21.33,30.67
ETHUSDT,0.03,4.0,8.05,20.05,28.11
LINKUSDT,0.54,4.0,9.08,21.08,30.16
LTCUSDT,0.69,4.0,9.39,21.39,30.78
SOLUSDT,0.20,4.0,8.40,20.40,28.79


median both-legs round trip: 29.8 bps  (plan placeholder assumed ~34)


In [17]:
# opportunity frequency: fraction of rows whose realized cum-24 clears k*roundtrip
feat=feat.merge(g[["symbol","both_rt"]],on="symbol",how="left"); feat["cum_bps"]=feat["realized_cum"]*1e4
r=[]
for y in sorted(feat.year.unique()):
    d=feat[feat.year==y]
    r.append({"year":y, **{f"k={k}":np.mean(d["cum_bps"]>k*d["both_rt"]) for k in (1,1.5,2,3)}})
r.append({"year":"ALL", **{f"k={k}":np.mean(feat["cum_bps"]>k*feat["both_rt"]) for k in (1,1.5,2,3)}})
pd.DataFrame(r).set_index("year").round(3)

,k=1,k=1.5,k=2,k=3
year,,,,
2022,0.000,0.000,0.000,0.000
2023,0.133,0.053,0.028,0.008
2024,0.266,0.161,0.096,0.052
ALL,0.135,0.072,0.042,0.020


In [18]:
# entry economics using causal OOF elastic: predicted>=k*rt, then realized net carry
o=oofc.merge(g[["symbol","both_rt"]],on="symbol",how="left")
o["pred_bps"]=o["pred_elastic"]*1e4; o["cum_bps"]=o["realized_cum"]*1e4
r=[]
for k in (1,1.5,2,3):
    sig=o[o["pred_bps"]>=k*o["both_rt"]]
    if len(sig)==0: continue
    net=sig["cum_bps"]-sig["both_rt"]
    r.append({"k":k,"signals":len(sig),"precision_cover_cost":np.mean(sig["cum_bps"]>sig["both_rt"]),
              "mean_net_bps":net.mean(),"frac_positive":np.mean(net>0)})
pd.DataFrame(r).set_index("k").round(2)

,signals,precision_cover_cost,mean_net_bps,frac_positive
k,,,,
1.0,2849,0.80,28.81,0.80
1.5,1074,0.93,44.13,0.93
2.0,474,0.98,57.70,0.98
3.0,28,1.00,78.78,1.00


Round trip is about 30 bps, two thirds of it the 10 bps spot taker fee. Carry clears it 0% of 2022, 13% of 2023, 27% of 2024. Given a signal the trade pays and precision rises with the entry multiple k. On that same subset persistence ranks as well as the model, so the entry does not need the model. The model earns its place choosing among symbols that clear the hurdle together.

## Causal prediction table

The table the strategy joins on. Past-only, per-fold scaler, expanding window, both models plus persistence, keyed by symbol and timestamp.

In [19]:
tab=oofc[["symbol","ts","fold","pred_elastic","pred_gbm","pred_persist"]].rename(
        columns={"pred_persist":"pred_persistence"}).sort_values(["symbol","ts"]).reset_index(drop=True)
outdir=REPO/"research"/"results"; outdir.mkdir(exist_ok=True)
tab.to_parquet(outdir/"oof_predictions.parquet", index=False)
print("wrote", len(tab), "rows ->", (outdir/"oof_predictions.parquet").relative_to(REPO))
tab.head()

wrote 22448 rows -> research/results/oof_predictions.parquet


,symbol,ts,fold,pred_elastic,pred_gbm,pred_persistence
0,ADAUSDT,2022-12-07 00:00:00+00:00,0,0.001581,0.001363,0.000740
1,ADAUSDT,2022-12-07 08:00:00+00:00,0,0.000835,0.000978,0.002209
2,ADAUSDT,2022-12-07 16:00:00+00:00,0,0.001376,0.001189,-0.000694
3,ADAUSDT,2022-12-08 00:00:00.006000+00:00,0,0.001519,0.001150,0.002400
4,ADAUSDT,2022-12-08 08:00:00+00:00,0,0.000931,0.000403,-0.000275


In [20]:
# GBM holds up under causal training far better than the elastic net
m=oofc.copy()
r=[]
for c,label in [("pred_elastic","elastic"),("pred_persist","persistence"),("pred_gbm","gbm")]:
    row={"model":label,"pooled":rank_ic(m["realized_cum"],m[c])}
    for y in sorted(m.year.unique()):
        gy=m[m.year==y]; row[str(y)]=rank_ic(gy["realized_cum"],gy[c])
    r.append(row)
pd.DataFrame(r).set_index("model").round(3)

,pooled,2022,2023,2024
model,,,,
elastic,0.692,0.762,0.556,0.784
persistence,0.734,0.432,0.660,0.790
gbm,0.778,0.741,0.694,0.829


Causally the elastic falls below persistence in 2023 and pooled, while GBM beats persistence every year. Trees capture something linear misses. The table carries both, so a strategy can run on either.

## Reads

- The 0.7 is not a purge leak. The purge is 8 days, matching the label.
- Two-sided training inflates it. Causal training drops pooled IC from about 0.79 to 0.69 and erases the timing lift.
- Above the cost hurdle the elastic ties persistence. Its lift is in near-zero rows nothing trades.
- The robust, causal, every-year edge is cross-sectional symbol selection.
- The frozen JSON is a Ridge on the full window with one miscalibrated scaler. Not for a backtest.
- Net carry is rankable with a small lift but clears the 30 bps round trip only about 13% of the time, 0% in 2022. A 2024-regime book.
- GBM is the only model that beats persistence causally every year.